In [ ]:
#@title Run this to load the datasets and set up the environment. { display-mode: "form" }
# Run this every time you open the spreadsheet
import sys
import os

class HiddenPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

with HiddenPrints():
  %load_ext autoreload
  %autoreload 2
  from collections import Counter
  from importlib.machinery import SourceFileLoader
  import numpy as np
  from os.path import join
  import warnings
  warnings.filterwarnings("ignore")
  import nltk
  nltk.download('punkt', quiet=True)
  import seaborn as sns
  import matplotlib

  from nltk.tokenize import sent_tokenize, word_tokenize
  from nltk.stem.wordnet import WordNetLemmatizer
  from nltk.stem.porter import *
  from nltk.corpus import stopwords
  nltk.download('stopwords', quiet=True)
  from sklearn.feature_extraction.text import CountVectorizer
  from sklearn.linear_model import LogisticRegression
  from sklearn.model_selection import train_test_split
  from sklearn.metrics import multilabel_confusion_matrix
  from sklearn import metrics
  import gdown
  import numpy

  import pandas as pd
  import matplotlib.pyplot as plt
  !pip install Keras-Preprocessing
  from keras_preprocessing.text import Tokenizer
  from keras_preprocessing.sequence import pad_sequences
  from keras.models import Sequential
  from keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
  from sklearn.model_selection import train_test_split
  from keras.callbacks import EarlyStopping
  from keras.layers import Dropout
  import re
  from nltk.corpus import stopwords
  from nltk import word_tokenize
  STOPWORDS = set(stopwords.words('english'))
  from bs4 import BeautifulSoup
  import string

  from sklearn.metrics import confusion_matrix

  !wget -q --show-progress 'https://storage.googleapis.com/inspirit-ai-data-bucket-1/Data/AI%20Scholars/Sessions%206%20-%2010%20(Projects)/Project%20-%20NLP%2BFinance/finance_test.csv'
  !wget -q --show-progress 'https://storage.googleapis.com/inspirit-ai-data-bucket-1/Data/AI%20Scholars/Sessions%206%20-%2010%20(Projects)/Project%20-%20NLP%2BFinance/finance_train.csv'

def plot_confusion_matrix(y_true, y_predicted):
  cm = metrics.confusion_matrix(y_true, y_predicted)
  print ("Plotting the Confusion Matrix")
  labels = ["Negative","Neutral","Positive"]
  df_cm = pd.DataFrame(cm,index =labels,columns = labels)
  fig = plt.figure(figsize=(7,6))
  res = sns.heatmap(df_cm, annot=True,cmap='Blues', fmt='g')
  plt.yticks([0.5,1.5,2.5], labels,va='center')
  plt.title('Confusion Matrix - TestData')
  plt.ylabel('True label')
  plt.xlabel('Predicted label')
  plt.show()
  plt.close()


def get_finance_train():
  df_train = pd.read_csv("finance_train.csv")
  return df_train
def get_finance_test():
  df_test = pd.read_csv("finance_test.csv")
  return df_test

PUNCTUATION = '!#$%&()*,-./:;<=>?@^_`{|}~'

REPLACE_BY_SPACE_RE = re.compile('[/(){}\[\]\|@,;]')
BAD_SYMBOLS_RE = re.compile('[^0-9a-z #+_]')
STOPWORDS = set(stopwords.words('english'))
MAX_SEQUENCE_LENGTH = 256


def clean_text(text):
    """
        text: a string
        return: modified initial string
    """
    text = text.lower() # lowercase text
    text = REPLACE_BY_SPACE_RE.sub(' ', text) # replace REPLACE_BY_SPACE_RE symbols by space in text. substitute the matched string in REPLACE_BY_SPACE_RE with space.
    text = BAD_SYMBOLS_RE.sub('', text) # remove symbols which are in BAD_SYMBOLS_RE from text. substitute the matched string in BAD_SYMBOLS_RE with nothing.
    # text = text.replace('x', '')
#    text = re.sub(r'\W+', '', text)
    text = ' '.join(word for word in text.split() if word not in STOPWORDS) # remove stopwords from text
    return text

def pad_sequences_train(df_train, df_test, maxlen=MAX_SEQUENCE_LENGTH):
  tokenizer = Tokenizer(num_words=MAX_NB_WORDS+1, filters='!"#$%&()*+,-./:;<=>?@[\]^_`{|}~', lower=True)
  tokenizer.fit_on_texts(df_train['Sentence'].values)
  word_index = tokenizer.word_index
  X = tokenizer.texts_to_sequences(df_train['Sentence'].values)
  X_train = pad_sequences(X, maxlen=maxlen)
  return X_train

def pad_sequences_test(df_train, df_test, maxlen=MAX_SEQUENCE_LENGTH):
  tokenizer = Tokenizer(num_words=MAX_NB_WORDS+1, filters='!"#$%&()*+,-./:;<=>?@[\]^_`{|}~', lower=True)
  tokenizer.fit_on_texts(df_train['Sentence'].values)
  word_index = tokenizer.word_index
  X = tokenizer.texts_to_sequences(df_test['Sentence'].values)
  X_test = pad_sequences(X, maxlen=maxlen)
  return X_test

df_train2 = get_finance_train()
df_test2 = get_finance_test()
df_train2['Sentence'] = df_train2['Sentence'].apply(clean_text)
df_test2['Sentence'] = df_test2['Sentence'].apply(clean_text)

def run_model(X_train, y_train, X_test, y_test, epochs=5, batch_size=64, max_sequence_length=256, max_nb_words=1000, embedding_dim=300):
  if any(x is None for x in [X_train, y_train, X_test, y_test, epochs, max_sequence_length, max_nb_words, embedding_dim]):
    print('Replace the None values above with your new values before calling the run_model() function.')
    return None, None, None, None, None

  X_train2 = pad_sequences_train(df_train2, df_test2, max_sequence_length)
  X_test2 = pad_sequences_test(df_train2, df_test2, max_sequence_length)
  model = Sequential()
  model.add(Embedding(max_nb_words+1, embedding_dim, mask_zero=False, input_length=max_sequence_length))
  model.add(SpatialDropout1D(0.2))
  model.add(LSTM(100, dropout=0.2))
  model.add(Dense(n_labels, activation='softmax'))
  model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
  model.summary()
  history = model.fit(X_train2,
                    y_train,
                    epochs=epochs,
                    batch_size=batch_size,
                    validation_split=0.1,
                    callbacks=[EarlyStopping(monitor='val_loss', patience=3, min_delta=0.0001)])
  test_loss, test_accuracy = model.evaluate(X_test, y_test)
  return model, history, test_accuracy, X_train2, X_test2


EMBEDDING_DIM = 300

plt.rcParams.update({'font.size': 22})

nltk.download('wordnet', quiet=True)

print ("Train & test files are loaded.")

In [ ]:
#@title Exercise. Brainstorm some ways you can capture word order in your model?
method_1 = '' #@param {type:"string"}
method_2 = '' #@param {type:"string"}

In [ ]:
df_train = get_finance_train()
df_test = get_finance_test()

In [ ]:
print(len(df_train))
df_train.head()

In [ ]:
print(len(df_test))
df_test.head()

In [ ]:
#@title Instructor Solution

# Find the index of the longest sentence using np.argmax and len
longest_sentence_index = np.argmax([len(x) for x in df_train['Sentence'].str.split(' ')])

# Get the longest sentence and its length using the index we found
longest_sentence = df_train.iloc[longest_sentence_index]['Sentence']
longest_sentence_length = len(longest_sentence.split(' '))

print(f'The longest sentence has length {longest_sentence_length} and is: \n\n{longest_sentence}')

In [ ]:
#@title Run this cell to clean all the sentences in Finance Train and Finance Test.
#@markdown This will essentially remove our stopwords and punctuation marks.

df_train['Sentence'] = df_train['Sentence'].apply(clean_text)
df_test['Sentence'] = df_test['Sentence'].apply(clean_text)

In [ ]:
MAX_SEQUENCE_LENGTH = 256

In [ ]:
MAX_NB_WORDS = 1000

In [ ]:
#@title Instructor Solution

X_train = pad_sequences_train(df_train, df_test)
print(f'The shape of the dataframe is {X_train.shape}')
print(X_train[0])

In [ ]:
#@title Instructor Solution

X_test = pad_sequences_test(df_train, df_test)
print(f'The shape of the dataframe is {X_test.shape}')

In [ ]:
#@title Instructor Solution
y_train = pd.get_dummies(df_train['Label']).values
print(y_train.shape)

In [ ]:
#@title Instructor Solution
y_test = pd.get_dummies(df_test['Label']).values
print(y_test.shape)

In [ ]:
n_labels = 3
label_map = {0 : "negative",
             1 : "neutral",
             2 : "positive"}

In [ ]:
#@title Run this cell to instantiate your model!

model = Sequential()
model.add(Embedding(MAX_NB_WORDS+1, EMBEDDING_DIM, mask_zero=True, input_length=MAX_SEQUENCE_LENGTH))
model.add(SpatialDropout1D(0.2))
model.add(LSTM(100, use_cudnn=False, dropout=0.2))
model.add(Dense(n_labels, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

In [ ]:
#@title Instructor Solution

epochs = 5
batch_size = 64

In [ ]:
history = model.fit(X_train,
                    y_train,
                    epochs=epochs,
                    batch_size=batch_size,
                    validation_split=0.1,
                    callbacks=[EarlyStopping(monitor='val_loss', patience=3, min_delta=0.0001)])

In [ ]:
#@title Instructor Solution

test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test set loss is {test_loss} and accuracy is {test_accuracy:.2%}')

In [ ]:
#@title Instructor Solution

training_loss = history.history['loss']

In [ ]:
#@title Instructor Solution

validation_loss = history.history['val_loss']

In [ ]:
#@title Instructor Solution

fig = plt.figure(figsize=(12,6))
plt.title('Loss over Time')
plt.xlabel('Epochs')
plt.ylabel('Loss')

plt.plot(training_loss, label='train')
plt.plot(validation_loss, label='validation')

plt.legend()
plt.show()

In [ ]:
#@title Instructor Solution

training_accuracy = history.history['accuracy']
validation_accuracy = history.history['val_accuracy']

In [ ]:
#@title Instructor Solution

fig = plt.figure(figsize=(12,6))
plt.title('Accuracy over Time')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')

plt.plot(training_accuracy, label='train')
plt.plot(validation_accuracy, label='validation')

plt.legend()
plt.show()

In [ ]:
# @title Instructor Solution
model, history, test_acc, X_train_repadded, X_test_repadded = run_model(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    epochs=6,
    batch_size=64,
    max_sequence_length=256,
    max_nb_words=1500,
    embedding_dim=280
)
print('New test accuracy is {}'.format(test_acc))

In [ ]:
#@title Instructor Solution

training_accuracy = history.history['accuracy']
validation_accuracy = history.history['val_accuracy']

fig = plt.figure(figsize=(12,6))
plt.title('Accuracy over Time')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')

plt.plot(training_accuracy, label='train')
plt.plot(validation_accuracy, label='validation')

plt.legend()
plt.show()

In [ ]:
# Predict on our test set
y_prob = model.predict(X_test_repadded)
y_pred = y_prob.argmax(axis = -1)
y_true = y_test.argmax(axis = -1)

# plot the confusion matrix!
plot_confusion_matrix(y_true, y_pred)